# **Reddit Scraper Notebook for HealthPH+**


# **Dependencies**

In [13]:
import requests
import time
import pandas as pd
from datetime import datetime
from urllib.parse import urlparse, parse_qs
import os
import re
import csv


In [14]:
from pathlib import Path
from typing import Any
from urllib.parse import quote_plus
import hashlib
import json

try:
    from playwright.async_api import (
        async_playwright,
        TimeoutError as PlaywrightTimeoutError,
    )
except ImportError as exc:
    raise ImportError(
        "playwright is not installed. Run: pip install playwright && playwright install chromium"
    ) from exc

In [15]:
print("✅ Libraries loaded successfully")

✅ Libraries loaded successfully


## **Reddit**

### Configuration

In [ ]:
# ── USER SETTINGS ──────────────────────────────────────────────────────────────
# Configure keyword CSV files from docs/keywords/.
# The scraper will run each keyword sequentially and wait 3 seconds between keywords.

LIMIT     = 50   # Posts per page (max 100)
MAX_PAGES = 10   # Number of pages to paginate through
REQUEST_DELAY_SECONDS = 2
KEYWORD_DELAY_SECONDS = 3
KEYWORD_BATCH_SIZE = 20
BATCH_COOLDOWN_SECONDS = 90
REQUEST_TIMEOUT_SECONDS = 30
MAX_429_RETRIES = 4
BASE_429_BACKOFF_SECONDS = 30

CWD = Path.cwd()
ROOT_DIR = next(
    (
        p
        for p in [CWD, *CWD.parents]
        if (p / "docs" / "keywords").exists() and (p / "data").exists()
    ),
    CWD,
)
KEYWORD_FILES = [
    ROOT_DIR / "docs/keywords/by_language/cebuano_keywords.csv",
    ROOT_DIR / "docs/keywords/by_language/filipino_keywords.csv",
    ROOT_DIR / "docs/keywords/by_language/hiligaynon_keywords.csv",
    ROOT_DIR / "docs/keywords/by_language/ilocano_keywords.csv",
]
KEYWORD_COLUMN_BY_FILE = {}
KEYWORD_COLUMNS = [KEYWORD_COLUMN_BY_FILE.get(filepath.name) for filepath in KEYWORD_FILES]
OUTPUT_FILE = str(ROOT_DIR / "data" / "raw" / "reddit" / "reddit_results.csv")   # Master data output file
# ───────────────────────────────────────────────────────────────────────────────

print(f"Keyword files: {len(KEYWORD_FILES)}")
for filepath, column in zip(KEYWORD_FILES, KEYWORD_COLUMNS):
    selected = column or "first column"
    print(f" - {filepath.name} ({selected})")
print(f"Limit      : {LIMIT} posts/page")
print(f"Max pages  : {MAX_PAGES}")
print(f"Page gap   : {REQUEST_DELAY_SECONDS} seconds")
print(f"Keyword gap: {KEYWORD_DELAY_SECONDS} seconds")
print(f"Batch pause: {BATCH_COOLDOWN_SECONDS} seconds after every {KEYWORD_BATCH_SIZE} keywords")
print(f"429 retry  : up to {MAX_429_RETRIES} retries starting at {BASE_429_BACKOFF_SECONDS} seconds")
print(f"Output file: {OUTPUT_FILE}")

In [17]:
## 3. Helper Functions
def extract_reddit_search_params(url):
    """
    Extract search query, sort order, and time filter from a Reddit search URL.
    
    Example URL:
        https://www.reddit.com/search/?q=sakit&sort=top&t=month
    
    Returns:
        tuple: (query, sort, time_filter)
    """
    parsed = urlparse(url)
    params = parse_qs(parsed.query)

    query = params.get("q", [None])[0]
    sort  = params.get("sort", ["relevance"])[0]
    t     = params.get("t",    ["all"])[0]

    if not query:
        raise ValueError(f"Could not extract a search query from URL: {url}")

    return query, sort, t


def _dedupe_keywords(values):
    keywords = []
    seen = set()
    for raw in values:
        chunks = [part.strip().strip('"').strip("'") for part in str(raw).split(",")]
        for chunk in chunks:
            if not chunk:
                continue
            key = chunk.lower()
            if key in seen:
                continue
            seen.add(key)
            keywords.append(chunk)
    return keywords


def load_keywords_from_csv(filepath, column=None):
    path = Path(filepath)
    if not path.exists():
        raise FileNotFoundError(f"Keyword file not found: {path}")

    if column is None:
        with path.open(newline="", encoding="utf-8") as csv_file:
            values = [cell for row in csv.reader(csv_file) for cell in row]
        return _dedupe_keywords(values)

    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    if df.empty:
        return []
    if column not in df.columns:
        raise ValueError(f"Column '{column}' not found in {path}. Available columns: {list(df.columns)}")
    return _dedupe_keywords(df[column].tolist())


def load_keywords_from_files(files, columns=None):
    if columns is None:
        columns = [None] * len(files)
    if len(files) != len(columns):
        raise ValueError(
            "files and columns must have the same length "
            f"(files={len(files)}, columns={len(columns)})"
        )

    merged = []
    seen = set()
    for filepath, column in zip(files, columns):
        for keyword in load_keywords_from_csv(filepath, column=column):
            key = keyword.lower()
            if key in seen:
                continue
            seen.add(key)
            merged.append(keyword)
    return merged


def build_search_url(keyword, sort="new", time_filter="all"):
    return (
        "https://www.reddit.com/search/?"
        f"q={quote_plus(keyword)}&sort={quote_plus(sort)}&t={quote_plus(time_filter)}"
    )


def _parse_retry_after_seconds(response, default_seconds):
    retry_after = response.headers.get("Retry-After")
    if not retry_after:
        return default_seconds

    try:
        return max(default_seconds, int(float(retry_after)))
    except (TypeError, ValueError):
        return default_seconds


def reddit_get_with_backoff(base_url, params, headers, timeout_seconds=30, max_retries=4, base_backoff_seconds=30):
    attempt = 0
    while True:
        response = requests.get(base_url, params=params, headers=headers, timeout=timeout_seconds)
        if response.status_code != 429:
            return response

        if attempt >= max_retries:
            print("❌ Reddit kept returning HTTP 429 after all retry attempts.")
            return response

        wait_seconds = _parse_retry_after_seconds(
            response,
            base_backoff_seconds * (2 ** attempt),
        )
        print(
            f"⚠️ HTTP 429 received. Cooling down for {wait_seconds} seconds "
            f"before retry {attempt + 1}/{max_retries}..."
        )
        time.sleep(wait_seconds)
        attempt += 1


def get_last_reddit_fullname_from_csv(filepath):
    """
    Read the last saved Reddit post URL from CSV and return its fullname (t3_<id>).

    Returns None if the file is missing, empty, or the URL is not parseable.
    """
    if not os.path.isfile(filepath):
        return None

    try:
        df = pd.read_csv(filepath, usecols=["url"])
        if df.empty:
            return None

        last_url = df["url"].dropna().iloc[-1]
        if not isinstance(last_url, str) or not last_url:
            return None

        match = re.search(r"/comments/([a-z0-9]+)/", last_url)
        if not match:
            return None

        return f"t3_{match.group(1)}"
    except Exception as e:
        print(f"⚠️ Could not read last id from {filepath}: {e}")
        return None


def scrape_reddit_search(url, limit=25, max_pages=3):
    """
    Scrape Reddit search results from a Reddit search URL.

    Args:
        url       : A Reddit search URL.
        limit     : Posts per page (max 100).
        max_pages : Maximum number of pages to paginate through.

    Returns:
        pd.DataFrame with columns: title, selftext, created, ups, subreddit, url
    """
    query, sort, t = extract_reddit_search_params(url)
    print(f"Query: '{query}'  |  Sort: {sort}  |  Time filter: {t}\n")

    headers  = {"User-Agent": "RedditScraper/1.0 (personal research script)"}
    base_url = "https://www.reddit.com/search.json"
    results  = []
    after    = None  # pagination cursor (within this run only)

    for page in range(max_pages):
        params = {
            "q":     query,
            "limit": limit,
            "sort":  sort,
            "t":     t,
            "type":  "link",   # posts only
        }
        if after:
            params["after"] = after

        response = reddit_get_with_backoff(
            base_url,
            params=params,
            headers=headers,
            timeout_seconds=REQUEST_TIMEOUT_SECONDS,
            max_retries=MAX_429_RETRIES,
            base_backoff_seconds=BASE_429_BACKOFF_SECONDS,
        )

        if response.status_code != 200:
            print(f"❌ Error: HTTP {response.status_code}")
            break

        data  = response.json()
        posts = data.get("data", {}).get("children", [])

        if not posts:
            print("ℹ️  No more posts found.")
            break

        for post in posts:
            pd_  = post.get("data", {})
            results.append({
                "title":     pd_.get("title", ""),
                "selftext":  pd_.get("selftext", ""),
                "created":   datetime.fromtimestamp(
                                 pd_.get("created_utc", 0)
                             ).strftime("%Y-%m-%d %H:%M:%S UTC"),
                "ups":       pd_.get("ups", 0),
                "subreddit": pd_.get("subreddit", ""),
                "url":       f"https://reddit.com{pd_.get('permalink', '')}",
            })

        after = data.get("data", {}).get("after")
        print(f"✔ Page {page + 1}: fetched {len(posts)} posts  (running total: {len(results)})")

        if not after:
            print("ℹ️  Reached last page.")
            break

        time.sleep(REQUEST_DELAY_SECONDS)   # polite delay to avoid rate limiting

    return pd.DataFrame(results)


def scrape_reddit_keywords(
    keywords,
    limit=25,
    max_pages=3,
    sort="new",
    time_filter="all",
    delay_seconds=3,
    batch_size=20,
    batch_cooldown_seconds=90,
):
    collected = []
    normalized_keywords = [keyword for keyword in keywords if str(keyword).strip()]

    for index, keyword in enumerate(normalized_keywords, start=1):
        print(f"[{index}/{len(normalized_keywords)}] Scraping keyword: {keyword}")
        search_url = build_search_url(keyword, sort=sort, time_filter=time_filter)
        keyword_df = scrape_reddit_search(url=search_url, limit=limit, max_pages=max_pages)
        if not keyword_df.empty:
            collected.append(keyword_df)

        if batch_size and index % batch_size == 0 and index < len(normalized_keywords):
            print(
                f"Completed {index} keywords. Cooling down for "
                f"{batch_cooldown_seconds} seconds before continuing..."
            )
            time.sleep(batch_cooldown_seconds)
        elif index < len(normalized_keywords):
            print(f"Sleeping {delay_seconds} seconds before the next keyword...")
            time.sleep(delay_seconds)

    if not collected:
        return pd.DataFrame(columns=["title", "selftext", "created", "ups", "subreddit", "url"])
    return pd.concat(collected, ignore_index=True)


def save_results(df, filepath=None):
    """
    Save Reddit posts to CSV without adding duplicates.
    Dedupe key: post URL.
    """
    if filepath is None:
        filepath = OUTPUT_FILE

    os.makedirs(os.path.dirname(filepath), exist_ok=True)

    if df.empty:
        print("ℹ️ No rows to save.")
        return

    if "url" not in df.columns:
        raise ValueError("save_results requires a 'url' column for deduplication.")

    file_exists = os.path.isfile(filepath)
    incoming_count = len(df)

    to_save = df.copy()
    to_save["url"] = to_save["url"].astype(str).str.strip()
    to_save = to_save[to_save["url"] != ""]

    # Remove duplicates in this scrape batch
    to_save = to_save.drop_duplicates(subset=["url"], keep="first")

    # Remove rows already present in the output file
    if file_exists and not to_save.empty:
        try:
            existing_urls = set(
                pd.read_csv(filepath, usecols=["url"])["url"]
                .dropna()
                .astype(str)
                .str.strip()
            )
            to_save = to_save[~to_save["url"].isin(existing_urls)]
        except ValueError:
            print("⚠️ Existing file has no 'url' column. Skipping cross-run dedupe.")

    new_rows = len(to_save)
    if new_rows == 0:
        skipped = incoming_count
        print(f"ℹ️ No new posts to append. (Skipped {skipped} duplicates)")
    else:
        to_save.to_csv(
            filepath,
            mode="a",
            index=False,
            encoding="utf-8-sig",
            header=not file_exists,
        )
        skipped = incoming_count - new_rows
        action = "Appended to" if file_exists else "Created"
        print(f"💾 {action} {filepath}  (+{new_rows} posts, skipped {skipped} duplicates)")

    total = pd.read_csv(filepath).shape[0] if os.path.isfile(filepath) else 0
    print(f"📊 Total rows in file: {total}")


print("✅ Functions defined")

✅ Functions defined


### Main Function

In [ ]:
keywords = load_keywords_from_files(KEYWORD_FILES, KEYWORD_COLUMNS)
print(f"Loaded {len(keywords)} unique keywords")

df = scrape_reddit_keywords(
    keywords,
    limit=LIMIT,
    max_pages=MAX_PAGES,
    sort="new",
    time_filter="all",
    delay_seconds=KEYWORD_DELAY_SECONDS,
    batch_size=KEYWORD_BATCH_SIZE,
    batch_cooldown_seconds=BATCH_COOLDOWN_SECONDS,
)
print(f"Total posts collected: {len(df)}")


### Results Preview 

In [19]:
# First 5 rows
df.head()

,title,selftext,created,ups,subreddit,url
0,Bulakenyong Tagalog,Pansin ko ang NCR iba ang gamit ng Tagalog mad...,2025-12-19 14:35:32 UTC,42,Tagalog,https://reddit.com/r/Tagalog/comments/1pqe7vl/...
1,When Do I Know If I Need My Wisdom Tooth Checked,This is how they look. Yung left yung medyo bo...,2025-08-24 11:44:01 UTC,1,DentistPh,https://reddit.com/r/DentistPh/comments/1mylfo...
2,nawawalan na ako ng pag asa,"sa 3 days before exam need ko mag aral, pero l...",2024-08-04 16:02:38 UTC,9,MedTechPH,https://reddit.com/r/MedTechPH/comments/1ejq3i...
3,"Got bitten by a dog, my sister makes it all ab...","I 27F, got bitten last night by one of our dog...",2024-07-17 23:19:14 UTC,119,OffMyChestPH,https://reddit.com/r/OffMyChestPH/comments/1e5...


In [20]:
# Upvote distribution
df["ups"].describe()

count      4.000000
mean      42.750000
std       53.841589
min        1.000000
25%        7.000000
50%       25.500000
75%       61.250000
max      119.000000
Name: ups, dtype: float64

In [21]:
# Top 10 posts by upvotes
df.sort_values("ups", ascending=False)[["title", "subreddit", "ups", "created"]].head(10)

,title,subreddit,ups,created
3,"Got bitten by a dog, my sister makes it all ab...",OffMyChestPH,119,2024-07-17 23:19:14 UTC
0,Bulakenyong Tagalog,Tagalog,42,2025-12-19 14:35:32 UTC
2,nawawalan na ako ng pag asa,MedTechPH,9,2024-08-04 16:02:38 UTC
1,When Do I Know If I Need My Wisdom Tooth Checked,DentistPh,1,2025-08-24 11:44:01 UTC


In [22]:
# Post count by subreddit
df["subreddit"].value_counts().head(10)

subreddit
Tagalog         1
DentistPh       1
MedTechPH       1
OffMyChestPH    1
Name: count, dtype: int64

### Save to CSV

In [23]:
save_results(df, OUTPUT_FILE)

💾 Appended to /Users/angelodelapaz/Documents/GitHub/healthphpersonal/data/raw/reddit/reddit_results.csv  (+4 posts, skipped 0 duplicates)
📊 Total rows in file: 3952


In [24]:
df.sample(10)

ValueError: Cannot take a larger sample than population when 'replace=False'